# Fund classification pipeline: who are the 152 hedge funds?

Addresses the discussant's slide 9 (Sokolovski, EFA DT 2026): show that the Cayman entities in the repo
data are overwhelmingly **fixed-income / rates relative-value and macro vehicles**, so their repo positions
reflect sovereign-bond trades rather than funding for unrelated strategies.

**Inputs** (all public except the LEI list): `hf_Valeri.xlsx` (LEI + name), the GLEIF API, and the SEC's
bulk Form ADV data + IAPD brochures.

**Chain**: LEI → GLEIF (name, status, fund-manager relationship) → SEC Form ADV Schedule D 7.B.(1)
(match fund LEI → adviser, CRD, RIA/ERA) → Part 2A brochure (strategy text) → keyword classification.

**Outputs** (written to `fund_classification/`, git-ignored): `fund_profiles.csv` (one row per fund with
every evidence column), `advisers.csv`, `summary.csv`, a LaTeX tabular for the paper, and a manual-review list.

**How to run**: needs internet (GLEIF + SEC are blocked from some research environments — run on a normal
machine). Set your contact email in `SEC_UA` below (the SEC requires an identifying User-Agent). Cells are
idempotent: everything downloaded is cached, re-running skips completed work. Runtime ≈ 5–10 min, dominated
by the GLEIF loop (rate limit ~1 request/sec) and the ADV zip downloads (~100–200 MB).

In [ ]:
import io, json, re, time, zipfile, difflib
from pathlib import Path

import pandas as pd
import requests

# repo root = folder containing hf_Valeri.xlsx (works whether the notebook runs from build/ or the root)
ROOT = Path.cwd() if (Path.cwd() / "hf_Valeri.xlsx").exists() else Path.cwd().parent
OUT = ROOT / "fund_classification"
for sub in ["cache/gleif", "sec", "brochures"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

# EDIT ME: the SEC requires a descriptive User-Agent with a contact address
SEC_UA = {"User-Agent": "JMP fund identification research your.name@your-university.example"}


# ---- network setup: corporate proxy + TLS interception (edit PROXY if needed) ----
# If you sit behind a corporate proxy, plain requests times out even though the
# browser works. Auto-detect what Windows knows; override manually if that is empty
# (find it under Settings -> Network & Internet -> Proxy, or ask IT).
import urllib.request

PROXY = ""            # EDIT if auto-detection fails, e.g. "http://proxy.mycorp.net:8080"

SESSION = requests.Session()
SESSION.headers.update(SEC_UA)
_auto = urllib.request.getproxies()
if PROXY:
    SESSION.proxies = {"http": PROXY, "https": PROXY}
elif _auto:
    SESSION.proxies = _auto
try:                   # corporate TLS inspection: trust the Windows certificate store
    import truststore  # pip install truststore  (harmless if absent)
    truststore.inject_into_ssl()
except ImportError:
    pass
print("proxies:", SESSION.proxies or "none (direct connection)")

# connectivity self-test: fail loudly here, not deep inside the GLEIF loop
try:
    _t = SESSION.get("https://api.gleif.org/api/v1/lei-records?page[size]=1", timeout=15)
    print("GLEIF reachable:", _t.status_code)
except Exception as e:
    print("GLEIF NOT reachable:", type(e).__name__, "-", e)
    print("-> set PROXY above to your corporate proxy, or run this notebook on a machine")
    print("   with normal internet access (only the xlsx + this notebook are needed).")

funds = pd.read_excel(ROOT / "hf_Valeri.xlsx").rename(columns={"entity_id": "lei"})
funds["lei"] = funds["lei"].str.strip().str.upper()
assert funds["lei"].str.fullmatch(r"[A-Z0-9]{18}[0-9]{2}").all(), "non-LEI entries in the input"
print(f"{len(funds)} funds, {funds['name'].isna().sum()} without a name in the extract")

## 1. GLEIF: registration record + fund-manager relationship

Free public registry, no key. Fills the missing names, and the Level-2 relationship record
(`IS_FUND-MANAGED_BY`) gives the **management entity** — the discussant's manager-level unit.
Results are cached per LEI, so re-runs are instant.

In [ ]:
GLEIF = "https://api.gleif.org/api/v1/lei-records"

def gleif_get(url, cache_file):
    if cache_file.exists():
        return json.loads(cache_file.read_text())
    r = SESSION.get(url, timeout=30)
    time.sleep(1.05)                                # stay under the ~60/min rate limit
    data = r.json() if r.status_code == 200 else {"_status": r.status_code}
    cache_file.write_text(json.dumps(data))
    return data

rows = []
for i, lei in enumerate(funds["lei"], 1):
    rec = gleif_get(f"{GLEIF}/{lei}", OUT / "cache/gleif" / f"{lei}.json")
    a = rec.get("data", {}).get("attributes", {})
    ent = a.get("entity", {})
    row = {
        "lei": lei,
        "gleif_name": (ent.get("legalName") or {}).get("name"),
        "gleif_status": ent.get("status"),
        "gleif_category": ent.get("category"),
        "gleif_jurisdiction": ent.get("jurisdiction"),
        "manager_lei": None, "manager_name": None,
    }
    # follow the fund-manager relationship if GLEIF reports one
    rel = rec.get("data", {}).get("relationships", {})
    link = next((v.get("links", {}).get("related")
                 for k, v in rel.items() if "fund-manager" in k and isinstance(v, dict)), None)
    if link:
        mgr = gleif_get(link, OUT / "cache/gleif" / f"{lei}_mgr.json")
        m = mgr.get("data", {})
        if isinstance(m, dict) and m.get("id"):
            row["manager_lei"] = m["id"]
            row["manager_name"] = (m.get("attributes", {}).get("entity", {})
                                     .get("legalName") or {}).get("name")
    rows.append(row)
    if i % 25 == 0:
        print(f"{i}/{len(funds)}")

gleif_df = pd.DataFrame(rows)
funds = funds.merge(gleif_df, on="lei", how="left")
funds["name"] = funds["name"].fillna(funds["gleif_name"])
print(funds[["gleif_status", "gleif_category"]].value_counts(dropna=False))
print(f"\nfund-manager relationship found for {funds['manager_name'].notna().sum()} funds")
print(funds["manager_name"].value_counts().head(20))

## 2. Entity-name classification

Many vehicles carry the strategy in their legal name ("Millennium **Fixed Income**", "Garda **FIRV**",
"Capula Global **Relative Value**"). This classifies the *sleeve*, which is the right unit here: a
multi-strategy manager's dedicated FI entity is fixed-income evidence, not multi-strategy noise.

In [ ]:
RULES = [
    ("Fixed income / rates RV", [
        "FIXED INCOME", "FIRV", "RELATIVE VALUE", " RATES", "G-10", "GLOBAL RATES",
        "INFLATION", "BOND", "TERM CREDIT", "CONVEX", "TAIL RISK", "VOLATILITY",
    ]),
    ("Global macro",  ["MACRO", "ALL WEATHER", "PURE ALPHA", "OPTIMAL PORTFOLIO", "DMO"]),
    ("Credit",        ["CREDIT", "ABS ", "HIGH YIELD", "DISTRESSED"]),
    ("Equity",        ["EQUITY"]),
    ("Commodity",     ["COMMODITY"]),
    ("Multi-strategy platform", ["MULTI-STRATEGY", "MULTI STRATEGY", "DIVERSIFIED ALPHA"]),
]

def classify_text(text):
    if not isinstance(text, str) or not text.strip():
        return None
    t = " " + re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9\- ]", " ", text.upper())) + " "
    for label, kws in RULES:
        if any(k in t for k in kws):
            return label
    return None

funds["class_entity_name"] = funds["name"].map(classify_text)
funds["class_manager_name"] = funds["manager_name"].map(classify_text)
print(funds["class_entity_name"].value_counts(dropna=False))

## 3. SEC Form ADV bulk data → adviser per fund

The SEC publishes all Form ADV filings as monthly zips of CSVs (registered advisers *and* exempt
reporting advisers — large non-US managers file as ERAs). Schedule D 7.B.(1) is the private-fund
file and contains each fund's LEI, so the match is direct; a normalized-name match catches the rest.

If the automatic link-scrape below fails (the SEC reshuffles its pages occasionally), download the
latest "Registered Investment Adviser" and "Exempt Reporting Adviser" zips by hand from
<https://www.sec.gov/data-research/sec-markets-data/information-about-registered-investment-advisers-exempt-reporting-advisers>
into `fund_classification/sec/` and re-run from the next cell.

In [ ]:
SEC_PAGE = ("https://www.sec.gov/data-research/sec-markets-data/"
            "information-about-registered-investment-advisers-exempt-reporting-advisers")

zips = sorted((OUT / "sec").glob("*.zip"))
if not zips:
    html = SESSION.get(SEC_PAGE, timeout=60).text
    links = re.findall(r'href="([^"]+\.zip)"', html)
    links = ["https://www.sec.gov" + l if l.startswith("/") else l for l in links]
    # newest registered (ia) and exempt (era) archive
    for pat in ["ia", "era"]:
        cand = [l for l in links if pat in Path(l).name.lower()]
        if cand:
            url = sorted(cand)[-1]
            dest = OUT / "sec" / Path(url).name
            print("downloading", url)
            dest.write_bytes(SESSION.get(url, timeout=600).content)
    zips = sorted((OUT / "sec").glob("*.zip"))
print("archives:", [z.name for z in zips])

In [ ]:
def read_csvs(zpath, name_contains):
    """Every CSV in the archive whose filename contains name_contains, concatenated."""
    frames = []
    with zipfile.ZipFile(zpath) as z:
        for n in z.namelist():
            if n.lower().endswith(".csv") and name_contains.lower() in Path(n).name.lower():
                frames.append(pd.read_csv(z.open(n), dtype=str, encoding="latin-1",
                                          low_memory=False, on_bad_lines="skip"))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

pf, base = [], []
for z in zips:
    pf.append(read_csvs(z, "7B1"))         # Schedule D 7.B.(1): private funds
    base.append(read_csvs(z, "Base"))      # adviser base file (name, CRD)
pf = pd.concat(pf, ignore_index=True)
base = pd.concat(base, ignore_index=True)
print("private-fund rows:", len(pf), "| adviser rows:", len(base))
print("7B1 columns:", list(pf.columns)[:40])

In [ ]:
LEI_RE = re.compile(r"^[A-Z0-9]{18}[0-9]{2}$")

def detect_col(df, test, min_share=0.3, sample=2000):
    best, share = None, 0.0
    for c in df.columns:
        s = df[c].dropna().astype(str).str.strip().str.upper().head(sample)
        if len(s) == 0: continue
        sh = s.map(lambda x: bool(test(x))).mean()
        if sh > share: best, share = c, sh
    return best if share >= min_share else None

lei_col = detect_col(pf, LEI_RE.match)
fname_col = detect_col(pf, lambda x: ("FUND" in x or "MASTER" in x or "L.P" in x or "LTD" in x))
crd_col_pf = next((c for c in pf.columns if "CRD" in c.upper()), None)
crd_col_base = next((c for c in base.columns if "CRD" in c.upper()), None)
aname_col = next((c for c in base.columns if "LEGAL" in c.upper() or "BUSINESS" in c.upper() or c.upper().strip() in ("1A", "NAME")), base.columns[0])
print("detected -> LEI:", lei_col, "| fund name:", fname_col, "| CRD (7B1):", crd_col_pf,
      "| CRD (base):", crd_col_base, "| adviser name:", aname_col)

pf["_lei"] = pf[lei_col].astype(str).str.strip().str.upper()

# 1) direct LEI match
m = funds.merge(pf.drop_duplicates("_lei"), left_on="lei", right_on="_lei", how="left")
print("LEI matches:", m["_lei"].notna().sum(), "/", len(funds))

# 2) fallback: normalized-name match for the rest
def norm(s):
    s = re.sub(r"[^A-Z0-9 ]", " ", str(s).upper())
    s = re.sub(r"\b(THE|LTD|LIMITED|LP|L P|LLC|LDC|INC|FUND|MASTER|CAYMAN)\b", " ", s)
    return re.sub(r"\s+", " ", s).strip()

pf["_norm"] = pf[fname_col].map(norm)
lookup = pf.drop_duplicates("_norm").set_index("_norm")
todo = m["_lei"].isna() & m["name"].notna()
for i in m.index[todo]:
    n = norm(m.at[i, "name"])
    hit = n if n in lookup.index else next(iter(difflib.get_close_matches(n, lookup.index, 1, 0.92)), None)
    if hit is not None:
        for c in pf.columns:
            m.at[i, c] = lookup.at[hit, c]
        m.at[i, "_match"] = "name"
m.loc[m["_lei"].notna(), "_match"] = m.loc[m["_lei"].notna(), "_match"].fillna("lei")
print("total ADV matches:", m["_match"].notna().sum(), "/", len(funds))

In [ ]:
# attach adviser name + registration status via CRD
base["_crd"] = base[crd_col_base].astype(str).str.strip()
base_small = base.drop_duplicates("_crd")[["_crd", aname_col]].rename(columns={aname_col: "adviser_name"})
m["_crd"] = m[crd_col_pf].astype(str).str.strip()
m = m.merge(base_small, on="_crd", how="left")
m["iapd_link"] = "https://adviserinfo.sec.gov/firm/summary/" + m["_crd"]
m.loc[m["_match"].isna(), "iapd_link"] = None

advisers = (m.dropna(subset=["adviser_name"])
              .groupby(["adviser_name", "_crd", "iapd_link"]).size()
              .rename("n_funds").reset_index().sort_values("n_funds", ascending=False))
advisers.to_csv(OUT / "advisers.csv", index=False)
print(advisers.head(30).to_string(index=False))

## 4. Part 2A brochures → strategy text

For each matched **registered** adviser, the strategy is described in the public Part 2A brochure
(Item 8). The cell tries to auto-download the newest brochure via the IAPD API; where that fails
(ERAs file no brochure; endpoints change), it prints the adviser's IAPD link — open the "Part 2
Brochures" tab and drop the PDF into `fund_classification/brochures/<CRD>.pdf`. The parser below
picks up whatever PDFs are in that folder, however they got there.

In [ ]:
def try_fetch_brochure(crd):
    dest = OUT / "brochures" / f"{crd}.pdf"
    if dest.exists():
        return "cached"
    for url in [f"https://api.adviserinfo.sec.gov/search/brochure/{crd}",
                f"https://api.adviserinfo.sec.gov/search/firm/{crd}"]:
        try:
            j = SESSION.get(url, timeout=30).json()
            ids = re.findall(r'"(?:brchrVrsnID|BRCHR_VRSN_ID|versionId)"\s*:\s*"?(\d+)', json.dumps(j))
            if ids:
                pdf = SESSION.get("https://files.adviserinfo.sec.gov/IAPD/Content/Common/"
                                   f"crd_iapd_Brochure.aspx?BRCHR_VRSN_ID={ids[0]}",
                                   headers=SEC_UA, timeout=60)
                if pdf.ok and pdf.content[:4] == b"%PDF":
                    dest.write_bytes(pdf.content)
                    return "downloaded"
        except Exception:
            pass
    return "manual"

status = {}
for crd, link in advisers[["_crd", "iapd_link"]].itertuples(index=False):
    status[crd] = try_fetch_brochure(crd)
    time.sleep(0.5)
print(pd.Series(status).value_counts())
print("\nmanual downloads needed (open link -> Part 2 Brochures tab):")
for crd, s in status.items():
    if s == "manual":
        print(" ", advisers.set_index("_crd").at[crd, "adviser_name"], "->",
              f"https://adviserinfo.sec.gov/firm/summary/{crd}")

In [ ]:
try:
    from pypdf import PdfReader
except ImportError:
    from PyPDF2 import PdfReader   # fallback

BRO_RULES = [
    ("Fixed income / rates RV", ["fixed income", "relative value", "sovereign", "government bond",
                                 "interest rate", "rates strateg", "repurchase agreement", "bond futures"]),
    ("Global macro",  ["global macro", "macroeconomic"]),
    ("Credit",        ["credit", "high yield", "distressed", "structured credit"]),
    ("Equity",        ["equity", "equities", "stock selection"]),
    ("Multi-strategy platform", ["multi-strategy", "multiple strategies", "trading pods"]),
]

def classify_brochure(pdf_path):
    try:
        text = " ".join((p.extract_text() or "") for p in PdfReader(str(pdf_path)).pages).lower()
    except Exception as e:
        return None, f"unreadable: {e}"
    hits = {lab: sum(text.count(k) for k in kws) for lab, kws in BRO_RULES}
    best = max(hits, key=hits.get)
    return (best if hits[best] >= 3 else None), json.dumps(hits)

bro = {}
for p in (OUT / "brochures").glob("*.pdf"):
    bro[p.stem] = classify_brochure(p)
m["class_brochure"] = m["_crd"].map(lambda c: bro.get(c, (None, None))[0])
m["brochure_hits"] = m["_crd"].map(lambda c: bro.get(c, (None, None))[1])
print(m["class_brochure"].value_counts(dropna=False))

## 5. Final classification and outputs

Evidence precedence: **entity name** (the sleeve's own label) → **brochure** (adviser strategy text) →
**manager name**. Whatever remains goes on the manual-review list — with GLEIF manager and IAPD link
attached, each residual case is a one-minute lookup.

In [ ]:
m["class_final"] = (m["class_entity_name"]
                    .fillna(m["class_brochure"])
                    .fillna(m["class_manager_name"])
                    .fillna("Manual review"))

keep = ["lei", "name", "gleif_status", "manager_name", "adviser_name", "_crd", "_match",
        "iapd_link", "class_entity_name", "class_brochure", "class_manager_name", "class_final"]
profiles = m[keep].rename(columns={"_crd": "adviser_crd", "_match": "adv_match_type"})
profiles.to_csv(OUT / "fund_profiles.csv", index=False)

summary = profiles["class_final"].value_counts().rename_axis("strategy").rename("n_funds").reset_index()
summary["share"] = (100 * summary["n_funds"] / len(profiles)).round(1)
summary.to_csv(OUT / "summary.csv", index=False)
print(summary.to_string(index=False))

print("\n--- LaTeX ---")
print("\\begin{tabular}{lrr}\n\\toprule\nStrategy & Funds & Share (\\%) \\\\\n\\midrule")
for _, r in summary.iterrows():
    print(f"{r['strategy']} & {r['n_funds']} & {r['share']} \\\\")
print("\\bottomrule\n\\end{tabular}")

print("\n--- manual review ---")
print(profiles.loc[profiles["class_final"] == "Manual review",
                   ["name", "manager_name", "adviser_name", "iapd_link"]].to_string(index=False))

**Volume weighting (ECB-side):** to report shares of *repo volume* rather than fund counts, merge
`fund_profiles.csv` with the internal per-fund volumes (the `hf.xlsx` produced by `build_main_panel.ipynb`)
on `lei` = `entity_id`, then aggregate `volume` by `class_final`. The volumes never leave the ECB
environment; only the aggregated shares go into the paper.